# 08 — Census disparate-impact fairness audit (Chicago / NYC / LA)

- **Owner:** Bella  
- **Date:** 2026-07-12 (refreshed 2026-07-19)  
- **Models audited:** Model 1 (risk) — full battery; Model 2 (forecast) — calibration + coverage.  
- **Basis:** each city's chronological **test split** with realised labels (not the forward-looking `scores.json`).  
- **Artifacts written:** `reports/fairness/fairness_audit_<city>.json` + `reports/figures/fairness_*.png`.

> **Refresh note (2026-07-19):** Chicago reproduces exactly (it reads the frozen deployed feature snapshot); NYC and LA re-pull current SODA, so their row counts and numbers move as the data grows. On this run LA surfaces a new neighborhood false-positive-rate finding, so the cities are not on a single as-of date. See `docs/fairness_audit.md`.

> **The one rule:** census demographics are **audit-only** — they measure disparate impact and are never a model feature (decisions 0004 / 0005). This notebook joins them *after* the model, never before.

The audit reads a city-agnostic `AuditFrame` from a per-city adapter, joins ACS tract demographics (`foodsafety.audit.census`), and runs the metrics engine (`foodsafety.audit.fairness`): flag-rate parity, FPR, FNR, and calibration by group, each with bootstrap CIs and a material-and-confident verdict. See `src/foodsafety/audit/README.md` for the design.

Requires `CENSUS_API_KEY` in the environment and the `audit` extra (`uv sync --extra audit`).

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import average_precision_score

from foodsafety.audit import census, fairness, report, mitigation
from foodsafety.audit.adapters.chicago import ChicagoAdapter
from foodsafety.audit.adapters.nyc import NycAdapter
from foodsafety.audit.adapters.la import LaAdapter
from foodsafety.audit.census import ACS_YEAR

pd.set_option('display.width', 200, 'display.max_columns', 30)
OUT = Path('..') / 'reports' / 'fairness'
OUT.mkdir(parents=True, exist_ok=True)
ADAPTERS = {'chicago': ChicagoAdapter(), 'nyc': NycAdapter(), 'la': LaAdapter()}

## Build each city's audit frame, join census, and write the report
Each adapter reproduces its city's deployed model on the test split; the census join adds tract demographics; `report.build_report` assembles the reviewable JSON.

In [ ]:
reports = {}
for city, adapter in ADAPTERS.items():
    frame = adapter.build_audit_frame()
    frame = census.attach_area_demographics(frame, city=city)
    rep = report.build_report(frame, city, acs_year=ACS_YEAR)
    (OUT / f'fairness_audit_{city}.json').write_text(json.dumps(rep, indent=2))
    reports[city] = rep
    p = rep['provenance']
    print(f"{city:8s} rows={p['test_rows']:>6}  prevalence={p['label_prevalence']:.3f}  "
          f"M1 PR-AUC={p['model1_test_pr_auc']:.3f}  window={p['test_window']}")

## Visual summary of the fairness metrics
The figures below read the freshly computed `reports` dict. Each is saved to `reports/figures/` for the writeup. Colors are colorblind-safe and every finding carries a non-color cue (a marker or hatch) as well as color.

In [ ]:
# Fairness-audit figures — colorblind-safe (Okabe-Ito); status is never
# color-alone (findings also carry a hatch and a marker). Figures are saved
# to reports/figures/ and displayed inline.
from pathlib import Path
FIGDIR = Path("..") / "reports" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np

# --- style -----------------------------------------------------------------
CITY_ORDER = ["chicago", "nyc", "la"]
CITY_LABEL = {"chicago": "Chicago", "nyc": "New York City", "la": "Los Angeles"}

# Okabe-Ito colorblind-safe palette.
OK = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "vermillion": "#D55E00",
    "sky": "#56B4E9",
    "purple": "#CC79A7",
    "grey": "#7F7F7F",
}
CLEAR = OK["blue"]        # within tolerance
FINDING = OK["vermillion"]  # material + CI-confident gap
INK = "#222222"
MUTED = "#6b6b6b"

# Human-readable axis names (the raw keys are terse).
AXIS_LABEL = {
    "neighborhood": "Neighborhood",
    "income": "Area income (quartile)",
    "race_nonwhite": "Area % non-white (quartile)",
    "race_dominant": "Area majority group",
    "poverty": "Area % in poverty (quartile)",
    "foreign_born": "Area % foreign-born (quartile)",
    "limited_english": "Area % limited-English (quartile)",
    "cuisine": "Cuisine",
    "tenure": "Establishment tenure",
    "facility_type": "Facility type",
}

LENS = [("fpr_gap", "False-positive-rate gap"),
        ("fnr_gap", "False-negative-rate gap"),
        ("ece_gap", "Calibration-error gap")]
LENS_COLOR = {"fpr_gap": OK["blue"], "fnr_gap": OK["green"], "ece_gap": OK["orange"]}


def _apply_style():
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#cccccc",
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "grid.color": "#e8e8e8",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.labelsize": 11,
        "xtick.color": INK, "ytick.color": INK, "text.color": INK,
        "axes.labelcolor": INK,
        "svg.fonttype": "none",
    })


def _axis_label(key):
    return AXIS_LABEL.get(key, key.replace("_", " ").title())


def _gap(ax_data, metric, point="gaps_primary_high"):
    """Return the {value, ci_low, ci_high, finding, tolerance} entry for a metric."""
    for g in ax_data.get(point, []):
        if g["metric"] == metric:
            return g
    return None


# Short labels for the truth-conditioned metrics (used in annotations).
_METRIC_SHORT = {"fpr_gap": "false-positive", "fnr_gap": "false-negative", "ece_gap": "calibration"}


def fig_findings_by_lens(reports):
    """How many axes fire on the parity lens vs any truth-conditioned lens.

    Fully data-driven: the truth-conditioned bars are annotated with the exact
    axis + lens that fired, whatever the current data shows.
    """
    _apply_style()
    fig, ax = plt.subplots(figsize=(9.5, 3.8))
    truth = {"fpr_gap", "fnr_gap", "ece_gap"}
    parity_counts, truth_counts, totals, truth_notes = [], [], [], []
    for city in CITY_ORDER:
        axes = reports[city]["model1_risk"]["axes"]
        p = t = 0
        notes = []
        for key, a in axes.items():
            fired = {g["metric"] for g in a.get("gaps_primary_high", []) if g["finding"]}
            if "disparate_impact_ratio" in fired:
                p += 1
            hits = fired & truth
            if hits:
                t += 1
                for m in hits:
                    notes.append(f"{_axis_label(key)} ({_METRIC_SHORT[m]})")
        parity_counts.append(p)
        truth_counts.append(t)
        totals.append(len(axes))
        truth_notes.append(notes)
    y = np.arange(len(CITY_ORDER))
    h = 0.36
    ax.barh(y + h / 2, parity_counts, height=h, color=OK["sky"],
            edgecolor="white", label="Parity gap (flag rate differs)")
    ax.barh(y - h / 2, truth_counts, height=h, color=FINDING, hatch="///",
            edgecolor="white", label="Truth-conditioned gap (FPR / FNR / calibration)")
    for yi, (p, t, n, notes) in enumerate(zip(parity_counts, truth_counts, totals, truth_notes)):
        ax.text(p + 0.1, yi + h / 2, f"{p} of {n} axes", va="center", fontsize=9.5, color=MUTED)
        label = f"{t}" + (f"  ← {', '.join(notes)}" if notes else "")
        ax.text(t + 0.1, yi - h / 2, label, va="center", fontsize=9, color=FINDING if t else MUTED)
    ax.set_yticks(y)
    ax.set_yticklabels([CITY_LABEL[c] for c in CITY_ORDER])
    ax.set_xlabel("Number of demographic axes with a finding")
    ax.set_xlim(0, max(totals) + 4)
    ax.set_title("Where do fairness findings land? Parity fires broadly; the bias lenses rarely")
    ax.legend(loc="lower right", fontsize=9, frameon=False)
    fig.tight_layout()
    return fig


# --- Fig 1: parity ratio by axis -------------------------------------------
def fig_parity(reports):
    """Disparate-impact (four-fifths) ratio per axis, with bootstrap CI, per city."""
    _apply_style()
    fig, axs = plt.subplots(1, 3, figsize=(15, 5.2), sharex=True)
    for col, city in enumerate(CITY_ORDER):
        ax = axs[col]
        axes = reports[city]["model1_risk"]["axes"]
        rows = []
        for key, a in axes.items():
            g = _gap(a, "disparate_impact_ratio")
            if g is None or g["value"] is None:
                continue
            rows.append((key, g["value"], g.get("ci_low"), g.get("ci_high"), g["finding"]))
        rows.sort(key=lambda r: r[1])  # worst parity at top
        labels = [_axis_label(r[0]) for r in rows]
        vals = [r[1] for r in rows]
        y = np.arange(len(rows))
        colors = [FINDING if r[4] else CLEAR for r in rows]
        hatches = ["///" if r[4] else "" for r in rows]
        bars = ax.barh(y, vals, color=colors, edgecolor="white", height=0.7)
        for b, hh in zip(bars, hatches):
            b.set_hatch(hh)
        # bootstrap CI whiskers
        for yi, r in enumerate(rows):
            lo, hi = r[2], r[3]
            if lo is not None and hi is not None:
                ax.plot([lo, hi], [yi, yi], color=INK, lw=1.1, alpha=0.6, zorder=5)
        # finding marker (non-color cue)
        for yi, r in enumerate(rows):
            if r[4]:
                ax.text(vals[yi] + 0.02, yi, "▲", va="center", ha="left",
                        fontsize=9, color=FINDING)
        ax.axvline(0.8, color=OK["vermillion"], ls="--", lw=1.3, zorder=1)
        ax.axvline(1.0, color=MUTED, ls=":", lw=1.1, zorder=1)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=9.5)
        ax.set_xlim(0, 1.12)
        ax.set_ylim(-0.7, len(rows) - 0.3)
        ax.set_title(CITY_LABEL[city])
        ax.set_xlabel("Disparate-impact ratio\n(lowest group flag rate ÷ highest)")
    fig.suptitle("Statistical parity by demographic axis  —  flagged = deployed High tier",
                 fontsize=13, fontweight="bold", y=0.99)
    fig.text(0.5, 0.93,
             "Shorter bar = larger flag-rate gap.   Dashed line = four-fifths rule (0.80);   "
             "dotted line = perfect parity (1.0).",
             ha="center", fontsize=9.5, color=MUTED)
    from matplotlib.patches import Patch
    handles = [Patch(facecolor=CLEAR, label="Within four-fifths rule"),
               Patch(facecolor=FINDING, hatch="///", label="Parity finding (▲ below 0.80, CI-confident)")]
    fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False,
               fontsize=10, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.05, 1, 0.9))
    return fig


# --- Fig 2: truth-conditioned lenses vs tolerance (forest plot) ------------
# A common axis-ordering so the three city panels line up row-for-row.
_LENS_AXIS_ORDER = ["neighborhood", "facility_type", "cuisine", "tenure",
                    "race_dominant", "race_nonwhite", "income", "poverty",
                    "foreign_born", "limited_english"]


def fig_bias_lenses(reports):
    """Forest plot: FPR / FNR / calibration gaps (as a fraction of their tolerance),
    each with its bootstrap CI. A finding only when the whole interval clears 1.0."""
    _apply_style()
    fig, axs = plt.subplots(1, 3, figsize=(15.5, 6.4))
    for col, city in enumerate(CITY_ORDER):
        ax = axs[col]
        axes = reports[city]["model1_risk"]["axes"]
        keys = [k for k in _LENS_AXIS_ORDER if k in axes]
        y = np.arange(len(keys))
        xmax = 1.3
        for j, (metric, _name) in enumerate(LENS):
            offs = (j - 1) * 0.24
            for yi, key in enumerate(keys):
                g = _gap(axes[key], metric)
                if g is None or g["value"] is None:
                    continue
                tol = g["tolerance"]
                val = g["value"] / tol
                lo = (g["ci_low"] or 0) / tol
                hi = (g["ci_high"] or 0) / tol
                xmax = max(xmax, hi * 1.05)
                ax.plot([lo, hi], [yi + offs, yi + offs], color=LENS_COLOR[metric],
                        lw=1.4, alpha=0.55, zorder=3, solid_capstyle="round")
                if g["finding"]:
                    ax.scatter(val, yi + offs, s=52, color=LENS_COLOR[metric],
                               edgecolor=INK, linewidth=1.3, zorder=5)
                    ax.text(hi + 0.05, yi + offs, "▲ finding", va="center",
                            fontsize=8, color=INK)
                else:
                    ax.scatter(val, yi + offs, s=26, color=LENS_COLOR[metric],
                               edgecolor="white", linewidth=0.6, zorder=4)
        ax.axvline(1.0, color=OK["vermillion"], ls="--", lw=1.3, zorder=1)
        ax.set_yticks(y)
        ax.set_yticklabels([_axis_label(k) for k in keys], fontsize=9.5)
        ax.set_ylim(-0.6, len(keys) - 0.4)
        ax.set_xlim(0, xmax)
        ax.set_title(CITY_LABEL[city])
        ax.set_xlabel("Gap ÷ its tolerance  (1.0 = the limit)")
    fig.suptitle("The bias lenses (false-positive, false-negative, calibration gaps) "
                 "vs their tolerance", fontsize=13, fontweight="bold", y=0.99)
    fig.text(0.5, 0.93, "Dot = estimate, line = 95% CI.   A finding requires the whole "
             "interval to clear the tolerance line: a dot past 1.0 whose line crosses back "
             "is not confident.",
             ha="center", fontsize=9.5, color=MUTED)
    from matplotlib.lines import Line2D
    handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=LENS_COLOR[m],
                      markersize=9, label=n) for m, n in LENS]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=10,
               bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=(0, 0.05, 1, 0.9))
    return fig


# --- Fig 3: prevalence tracking --------------------------------------------
def fig_prevalence_tracking(reports):
    """For parity-flagged axes: does the group flag rate track the real failure rate?"""
    _apply_style()
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    rows = []
    for city in CITY_ORDER:
        for key, a in reports[city]["model1_risk"]["axes"].items():
            g = _gap(a, "disparate_impact_ratio")
            if g and g["finding"] and a.get("flag_vs_prevalence_corr") is not None:
                rows.append((city, key, a["flag_vs_prevalence_corr"]))
    rows.sort(key=lambda r: r[2])
    y = np.arange(len(rows))
    corrs = [r[2] for r in rows]
    colors = [OK["green"] if c >= 0.5 else OK["orange"] for c in corrs]
    bars = ax.barh(y, corrs, color=colors, edgecolor="white", height=0.7)
    for yi, r in enumerate(rows):
        mk = "✓ tracks risk" if r[2] >= 0.5 else "○ weak/none"
        ax.text(max(r[2], 0) + 0.02, yi, mk, va="center", fontsize=8,
                color=OK["green"] if r[2] >= 0.5 else OK["orange"])
    ax.axvline(0.5, color=MUTED, ls="--", lw=1.2)
    ax.text(0.5, -0.9, "prevalence-tracking\nthreshold (0.5)", color=MUTED,
            fontsize=8.5, va="top", ha="center")
    ax.set_yticks(y)
    ax.set_yticklabels([f"{CITY_LABEL[r[0]]}: {_axis_label(r[1])}" for r in rows], fontsize=9.5)
    ax.set_xlabel("Correlation of group flag rate with group failure rate")
    ax.set_xlim(min(0, min(corrs) - 0.1), 1.28)
    ax.set_ylim(-1.6, len(rows) - 0.3)
    ax.set_title("Do parity gaps track real risk?\n"
                 "High correlation = the model flags genuinely higher-risk areas, not bias",
                 fontsize=12)
    fig.tight_layout()
    return fig


# --- Fig 4: NYC cuisine calibration spotlight ------------------------------
def fig_nyc_cuisine(reports):
    """The one bias-lens finding: NYC calibration error across cuisines."""
    _apply_style()
    a = reports["nyc"]["model1_risk"]["axes"].get("cuisine")
    if a is None:
        return None
    gt = [r for r in a["group_table"] if r.get("audited") and r.get("ece") is not None]
    gt.sort(key=lambda r: r["ece"])
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(14.5, 6.2))
    tol = reports["nyc"]["tolerances"]["ece_gap_max"]
    worst, best = gt[-1], gt[0]

    # left: ECE per cuisine vs tolerance; the max and min define the audited gap.
    y = np.arange(len(gt))
    bar_colors = [FINDING if r is worst else (OK["green"] if r is best else OK["sky"])
                  for r in gt]
    axl.barh(y, [r["ece"] for r in gt], color=bar_colors, edgecolor="white", height=0.72)
    axl.set_yticks(y)
    axl.set_yticklabels([r["group"] for r in gt], fontsize=9)
    axl.set_xlabel("Expected calibration error\n(|mean predicted − mean observed| risk)")
    axl.set_title("Calibration error by cuisine")
    axl.set_xlim(0, worst["ece"] * 1.35)
    axl.annotate("", xy=(worst["ece"], len(gt) - 1), xytext=(best["ece"], 0),
                 arrowprops=dict(arrowstyle="<->", color=FINDING, lw=1.3, ls=(0, (4, 3))))
    axl.text(worst["ece"] * 1.02, len(gt) / 2,
             f"across-group gap\n= {worst['ece'] - best['ece']:.3f}\n(tolerance {tol})",
             color=FINDING, fontsize=9, va="center")

    # right: reliability — predicted vs observed per cuisine, zoomed to the data.
    xs = [r["mean_pred"] for r in gt]
    ys = [r["mean_obs"] for r in gt]
    for r in gt:
        is_worst = r is worst
        axr.scatter(r["mean_pred"], r["mean_obs"],
                    s=max(35, r["n"] / 12), color=FINDING if is_worst else OK["blue"],
                    edgecolor="white", zorder=5, alpha=0.85)
    # Label only the notable points (extremes + biggest miscalibrations) to avoid clutter.
    notable = {worst["group"], best["group"]}
    for r in sorted(gt, key=lambda r: -abs(r["mean_obs"] - r["mean_pred"]))[:4]:
        notable.add(r["group"])
    for r in gt:
        if r["group"] in notable:
            axr.annotate(r["group"], (r["mean_pred"], r["mean_obs"]), fontsize=8.5,
                         color=INK, xytext=(5, 4), textcoords="offset points")
    lo = min(min(xs), min(ys)) - 0.03
    hi = max(max(xs), max(ys)) + 0.05
    axr.plot([lo, hi], [lo, hi], ls="--", color=MUTED, lw=1.2, zorder=1)
    axr.text(hi, hi, "perfect\ncalibration", color=MUTED, fontsize=8.5, ha="right", va="top")
    axr.annotate("above the line:\nmodel under-predicts risk", xy=(lo + 0.02, hi - 0.02),
                 fontsize=8.5, color=MUTED, va="top")
    axr.set_xlim(lo, hi)
    axr.set_ylim(lo, hi)
    axr.set_xlabel("Mean predicted risk")
    axr.set_ylabel("Mean observed failure rate")
    axr.set_title("Reliability by cuisine (marker size ∝ group size)")
    fig.suptitle("New York City cuisine: the audit's only calibration finding",
                 fontsize=13, fontweight="bold", y=0.99)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig

def show(fig, name):
    """Save a figure to reports/figures/ and return it for inline display."""
    if fig is not None:
        fig.savefig(FIGDIR / f"{name}.png", dpi=130, bbox_inches="tight")
    return fig


### At a glance — where do findings land?
How many demographic axes fire on the **parity** lens (flag rate differs across groups) versus any **truth-conditioned** lens (false-positive, false-negative, or calibration gap). Parity fires often; the bias lenses are what actually catch unfair errors.

In [ ]:
show(fig_findings_by_lens(reports), 'fairness_00_findings_by_lens')

### Statistical parity by demographic axis
The disparate-impact (four-fifths) ratio: the lowest group's flag rate divided by the highest. 1.0 is perfect parity; below 0.80 (dashed line) is a potential concern. A shorter bar means a larger flag-rate gap. Whiskers are bootstrap 95% CIs; a finding (hatched, marked with a triangle) is both below 0.80 and CI-confident.

In [ ]:
show(fig_parity(reports), 'fairness_01_parity')

### The bias lenses vs their tolerance
False-positive-rate, false-negative-rate, and calibration-error gaps, each divided by its own tolerance so 1.0 is the limit for all three. The dot is the point estimate and the line is the 95% CI. A finding needs the **whole** interval past the line: a dot beyond 1.0 whose interval crosses back is not a confident finding.

In [ ]:
show(fig_bias_lenses(reports), 'fairness_02_bias_lenses')

### Do parity gaps track real risk?
For every axis with a parity finding, this shows the correlation between a group's flag rate and its actual failure rate. A high correlation (past 0.5) means the model flags genuinely higher-risk areas, not that it is biased.

In [ ]:
show(fig_prevalence_tracking(reports), 'fairness_03_prevalence_tracking')

### New York City cuisine — a calibration finding
NYC is the only city with a native cuisine field. **Left:** expected calibration error by cuisine, with the across-group gap that constitutes the finding. **Right:** predicted vs observed risk per cuisine; points above the diagonal are under-predicted.

In [ ]:
show(fig_nyc_cuisine(reports), 'fairness_04_nyc_cuisine')

## Per-city summary — which axes fire, on which lens
The key read: a **parity-only** finding (flag rate differs) is expected wherever true prevalence differs across groups — that is the model flagging higher-risk places, not bias. The lenses that catch bias are **FPR / FNR / calibration**, because they condition on the realised label.

In [ ]:
for city, rep in reports.items():
    print(f'==================== {city.upper()} ====================')
    print(pd.DataFrame(rep['model1_risk']['summary']).to_string(index=False))
    print()

## Verdict per axis (Model 1), with prevalence-tracking and the secondary operating point

In [ ]:
for city, rep in reports.items():
    print(f'-------------------- {city.upper()} --------------------')
    for key, ax in rep['model1_risk']['axes'].items():
        print(f"[{key}] corr={ax['flag_vs_prevalence_corr']}")
        print('   ', ax['verdict'])
    print()

## Model 2 (forecast) — calibration by group
The forecast has no flagging operating point, so only calibration + coverage are audited.

In [ ]:
for city, rep in reports.items():
    rows = [{'axis': k, 'coverage': a['coverage'], 'ece_gap': a['ece_gap'],
             'finding': a['finding']} for k, a in rep['model2_forecast']['axes'].items()]
    print(f'{city.upper()}:')
    print(pd.DataFrame(rows).to_string(index=False))
    print()

## Mitigation cost (analysis only)
If an equalized-odds gap appeared, this prices the per-group thresholds that equalize recall. It does **not** change the model — adopting per-group thresholds is a scope call (Jun).

In [ ]:
chi = census.attach_area_demographics(ADAPTERS['chicago'].build_audit_frame(), city='chicago')
tbl = mitigation.equalize_recall_thresholds(chi, 'area_income_q')
print(tbl.to_string(index=False))
print('extra inspections to equalize recall across income quartiles:',
      tbl.attrs['total_delta_flags'], 'of', tbl.attrs['baseline_flagged'], 'flagged')